In [14]:
import os
import sys
import json

DATA_DIR = '../claude_inference/runs/test_sqa50/'

# get starting and ending questions from all jsonl files in the directory (as pairs)
# {"kind": "run_start", "run_id": "sample_001", "timestamp": "2026-06-02T20:52:45.473449+00:00"}
# {"kind": "claude_call", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "purpose": "harder", "model": "claude-sonnet-4-5", "system": "You are an expert in constructing challenging research questions.\n\nGiven a seed question, produce an updated question designed to expose weaknesses in deep research systems. \n\nANSWERING SYSTEM PROFILE:\nThe answering system is an open \"deep research\" model that is trained to produce attributed long-form answers and whose ONLY tool is Semantic Scholar (\"S2\") search over academic papers. The system retrieves from a corpus of academic papers and synthesizes a cited report. Difficulty must not come from requiring sources outside this corpus. The system is good at surveying a single well-studied topic and producing a long well-structured report. It is bad at complex reasoning. \n\nOUTPUT FORMAT (valid JSON, no extra text):\n{\n  \"brainstorming\": \"<think about distinct strategies and reason about why they may or may not work>\",\n  \"chosen_strategy\": \"<name and justify the single most promising strategy>\",\n  \"updated_question\": \"<the rewritten question>\",\n  \"why_harder\": \"<explanation of why this question might be hard for a deep research system>\",\n  \"verification_criterion\": \"<one concrete, testable criterion for checking whether the answer is good>\"\n}\n\nRULES:\n- Avoid questions that can easily be answered by retrieving information.\n- The updated question should be hard to answer correctly, not just hard to retrieve \u2014 via higher-order thinking (analysis, comparison, evaluation, synthesis), a reasoning trap the system must catch (false premise, misconception, unanswerable claim), or an embedded constraint that changes what a correct answer must contain.\n- The updated question length should change by fewer than 15 words from the seed.\n- The verification criterion should be specific and checkable, not vague or aspirational. The criterion is checked by a judge who sees ONLY the question, the answer, and the answer's own cited sources \u2014 there is NO external answer key. So don't use hollow existence-counts like \"identify at least three implicit assumptions\" or \"name four categories of evidence.\" Anchor it to THIS question by naming the actual entities/claims at issue \u2014 never a generic template. \n- Select whichever strategy best fits THIS seed; do not default to the strategies shown in the examples below.\n\nEXAMPLE STRATEGIES TO CONSIDER:\n1. Require synthesis across 5+ sources or clearly disjoint domains (e.g., science + economics).\n2. Require synthesis across differing viewpoints, stakeholder incentives, or theoretical frameworks.\n3. Require multi-step reasoning, structured argumentation, or hierarchical planning.\n4. Require handling conflicting, incomplete, or low-quality evidence.\n5. Require correcting a hidden misconception or establishing key knowns before answering.\n6. Embed a specific context that changes the answer (e.g., \"explain to a policymaker with no ML background\").\n7. Make a question that is unanswerable by current research.\n\nHere are a few examples:\nSeed Question: What is pretraining-data deduplication?\n{\n\"brainstorming\": \"The current question can easily be answered by retrieving deduplication literature broadly. To make it harder, let's make the question unanswerable by asking a question that isn't answered by current literature.\",\n\"chosen_strategy\": \"Make the question unanswerable by asking a question that hasn't been resolved by current research.\",\n\"updated_question\": \"What is the causal contribution of pretraining-data deduplication to downstream reasoning, holding all else constant?\",\n\"why_harder\": \"A system can easily define deduplication, but it is much harder to determine its isolated causal effect on reasoning because existing studies do not cleanly vary only deduplication while holding other training factors fixed.\",\n\"verification_criterion\": \"Because no controlled study isolates this effect, the answer must EXPLICITLY state the question is unresolved by current research, name the specific missing evidence, and qualify any partial findings as correlational not causal. Fails if it asserts a confident causal answer or implies the literature resolves it.\"\n}\n\nSeed Question: What is the role of attention sparsity in efficient transformers?\n{\n\"brainstorming\": \"Surveying efficient-transformer literature would let the system define sparsity and list methods, so a pure synthesis question is too easy. One option is multi-step reasoning about FLOPs tradeoffs, but those numbers can be retrieved and quoted directly. A stronger option exploits that benchmark results for sparse attention genuinely conflict across papers: the system can locate both 'sparse wins' and 'sparse loses' results, but is bad at the reasoning needed to reconcile them via confounds.\",\n\"chosen_strategy\": \"Require reconciliation of conflicting evidence: force the system to explain WHY cited papers disagree rather than just report their results.\",\n\"updated_question\": \"When do sparse-attention transformers underperform dense baselines, and why do reported results conflict?\",\n\"why_harder\": \"A survey can enumerate sparse-attention methods and their headline numbers, but reconciling contradictory sparse-vs-dense comparisons requires identifying confounds (sequence length, task type, matched compute) that the papers themselves rarely make explicit, which is a reasoning task rather than a retrieval task.\",\n\"verification_criterion\": \"The answer must show that its OWN cited sources disagree (some reporting sparse >= dense, others sparse < dense) and attribute the conflict to at least one concrete confound such as sequence length, task type (long-range vs short-context), or matched compute budget. Fails if it issues a single uniform verdict, or if the papers it cites do not actually report conflicting sparse-vs-dense comparisons.\"\n}\n\nSeed Question: How does brown adipose tissue produce heat?\n{\n\"brainstorming\": \"The seed is a clean survey: retrieve BAT/UCP1 literature and summarize thermogenesis. A single-topic false premise (e.g. mislocating a function) fails, because if the corpus already frames it as a known misconception the system just retrieves the debunking. So embed a false CONJUNCTION whose refutation is not packaged anywhere: assert that (a) UCP1 drives ATP synthesis and (b) this powers shivering thermogenesis. Each underlying fact (UCP1 uncouples to make heat not ATP; BAT mediates NON-shivering thermogenesis) is documented separately as background, but no source refutes this composite because no one proposes it. A survey-strong, reasoning-weak system retrieves the facts yet writes fluently around the premise without noticing the contradiction.\",\n\"chosen_strategy\": \"False premise via conjunction of separately-documented facts.\",\n\"updated_question\": \"How does UCP1-driven ATP synthesis power shivering thermogenesis?\",\n\"why_harder\": \"Surveying BAT thermogenesis returns UCP1=uncoupling=heat and BAT=non-shivering as separate background facts, but nothing in the corpus is framed as refuting 'ATP-powered shivering.' A non-reasoning synthesis can therefore produce a fluent answer that silently honors the premise. Rejecting it requires conjoining two facts the literature never assembles against this claim: that UCP1 bypasses ATP synthase, and that it is the non-shivering pathway.\",\n\"verification_criterion\": \"The answer must reject BOTH embedded errors: (1) state that UCP1 uncouples oxidative phosphorylation and dissipates the proton gradient as heat rather than synthesizing ATP, i.e. it bypasses/short-circuits ATP synthase; and (2) state that UCP1/BAT mediates NON-shivering thermogenesis, which is distinct from and an alternative to shivering thermogenesis (skeletal-muscle contraction). Fails if it describes UCP1 as producing ATP, treats BAT/UCP1 as the mechanism of shivering, or answers fluently as though the premise were coherent.\"\n}\n", "messages": [{"role": "user", "content": "Seed question: \u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236"}], "response_text": "```json\n{\n  \"brainstorming\": \"The seed question '\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236' (blockchain consensus mechanisms) is extremely broad and invites a straightforward survey answer: define consensus, list PoW/PoS/PBFT/etc., describe their trade-offs. A pure synthesis across mechanisms is still too easy because the system can retrieve comparison tables directly. Multi-step reasoning about security proofs might work, but those are often formalized in papers. A stronger option: exploit that the Chinese-language academic corpus on blockchain is much sparser than English, yet S2 indexes primarily English papers. If I keep the question in Chinese, the system must either (a) find Chinese papers (few exist), (b) translate the question internally and search English (likely), or (c) fail. But difficulty from language mismatch violates the spirit since it's not about reasoning. So instead, rewrite in English but embed a reasoning trap: assert a false trade-off that SOUNDS plausible (e.g., 'finality speed always trades off against decentralization') which surveys repeat informally, but which counterexamples (like certain BFT-based designs) refute. However, those counterexamples are documented, so a retrieval-strong system might find them. A better trap: require reconciling WHY different consensus families (PoW vs. PoS vs. BFT) achieve 'security' via fundamentally different threat models (51% hashpower vs. 1/3 stake vs. Byzantine quorum), and ask which threat model applies to a HYBRID scenario the literature doesn't address. Actually, hybrid consensus (PoW+PoS) IS studied, so that's retrievable. Strongest option: pose an unanswerable causality question about a dimension (energy efficiency) where PoW vs. PoS comparisons exist but controlled experiments don't, forcing the system to recognize confounds or admit uncertainty rather than confidently claiming 'PoS is greener' as if it's proven causal.\",\n  \"chosen_strategy\": \"Unanswerable causality + conflicting evidence: Ask for the isolated causal effect of a consensus design choice where studies compare outcomes but don't control confounds, and where the question's framing assumes a clean answer exists.\",\n  \"updated_question\": \"What is the causal effect of switching from Proof-of-Work to Proof-of-Stake on network energy consumption, holding validator set size and transaction throughput constant?\",\n  \"why_harder\": \"Surveys can retrieve 'PoW uses more energy than PoS' as a headline claim, but no study cleanly isolates consensus mechanism while holding validator count and throughput fixed across both. Real-world PoW\u2192PoS transitions (Ethereum Merge) change multiple variables simultaneously (validator set, issuance, throughput targets), so confident causal attribution requires reasoning about confounds. A retrieval-strong system will cite energy comparisons but is unlikely to recognize they don't answer the controlled counterfactual, or to state the question is unanswerable with current evidence.\",\n  \"verification_criterion\": \"The answer must either (1) explicitly state that no study controls validator set size and throughput while varying only consensus mechanism, so the causal effect is not cleanly established, OR (2) if it claims a causal answer, cite a source that reports a matched-parameter experiment (same validator count, same TPS) comparing PoW vs. PoS energy. Fails if it confidently asserts 'PoS reduces energy by X%' using comparisons (e.g., Ethereum pre/post-Merge, Bitcoin vs. Cardano) where validator counts, throughput, or other parameters differ, without acknowledging those confounds prevent causal inference.\"\n}\n```", "usage": {"input_tokens": 17, "output_tokens": 845, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 1777}, "cost_usd": 0.013259100000000001, "latency_s": 26.107261416036636, "error": null, "run_id": "sample_001", "timestamp": "2026-06-02T20:53:11.675797+00:00"}
# {"kind": "research_call", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "url": "http://localhost:8007/ask", "request_json": {"question": "What is the causal effect of switching from Proof-of-Work to Proof-of-Stake on network energy consumption, holding validator set size and transaction throughput constant?"}, "response_status": 200, "response_body": {"answer_chars": 13667, "trace_present": true}, "latency_s": 99.59951554104919, "error": null, "run_id": "sample_001", "timestamp": "2026-06-02T20:54:51.276898+00:00"}
# {"kind": "claude_call", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "purpose": "judge", "model": "claude-sonnet-4-5", "system": null, "messages": [{"role": "user", "content": "You are an expert evaluator of deep research system outputs.\n\nYou will be given:\n- A research question\n- The verification criterion that defines what a good answer must do\n- The answer the research system produced\n\nYour job is to judge whether the answer satisfies the verification criterion, and to flag other issues you notice (factual errors, hallucinations, evasion, missing reasoning, structural problems, etc.) even if those issues are not part of the criterion.\n\nQUESTION:\nWhat is the causal effect of switching from Proof-of-Work to Proof-of-Stake on network energy consumption, holding validator set size and transaction throughput constant?\n\nVERIFICATION CRITERION:\nThe answer must either (1) explicitly state that no study controls validator set size and throughput while varying only consensus mechanism, so the causal effect is not cleanly established, OR (2) if it claims a causal answer, cite a source that reports a matched-parameter experiment (same validator count, same TPS) comparing PoW vs. PoS energy. Fails if it confidently asserts 'PoS reduces energy by X%' using comparisons (e.g., Ethereum pre/post-Merge, Bitcoin vs. Cardano) where validator counts, throughput, or other parameters differ, without acknowledging those confounds prevent causal inference.\n\nANSWER:\n<answer omitted: 13667 chars \u2014 see results file>\n\nOUTPUT FORMAT (valid JSON, no extra text):\n{\n  \"criterion_satisfied\": <true | false>,\n  \"criterion_reasoning\": \"<why the answer does or does not satisfy the criterion>\",\n  \"other_issues\": [\"<issue 1>\", \"<issue 2>\", ...],\n  \"summary\": \"<2-4 sentence overall summary of how the answer performed and what to push on next time>\",\n  \"verdict\": \"<PASSED | FAILED>\"\n}\n\nPASSED means: the criterion is satisfied AND there are no critical issues.\nFAILED means: the criterion is not satisfied OR there are serious problems (hallucinations, refusals, off-topic).\n"}], "response_text": "```json\n{\n  \"criterion_satisfied\": false,\n  \"criterion_reasoning\": \"The answer fails the verification criterion. It confidently asserts large energy reductions (10-100\u00d7, '10\u2013100\u00d7 lower than PoW', 'three orders of magnitude') from switching PoW to PoS while claiming to hold validator set size and TPS constant, but does NOT cite any study that actually performs a matched-parameter experiment with the same validator count AND same TPS comparing PoW vs PoS. Instead, it relies on: (1) cross-chain comparisons (Bitcoin vs PoS systems) where both validator counts and throughput differ dramatically, (2) Ethereum 2.0 beacon chain projections that model PoS only (not a PoW vs PoS comparison at matched parameters), and (3) methodological discussions about controlling for throughput/validators in principle, without citing an actual study that does so. The answer never acknowledges that these confounds prevent clean causal inference\u2014it instead presents these confounded comparisons as if they establish the causal effect, violating the criterion's requirement to either (1) state no such study exists, or (2) cite an actual matched-parameter experiment.\",\n  \"other_issues\": [\n    \"Misleading framing: The answer presents itself as providing 'a causal interpretation' and 'causal takeaway' while actually relying on observational cross-chain comparisons where multiple parameters differ simultaneously.\",\n    \"Conflation of evidence types: The answer treats modeling exercises (Ethereum 2.0 energy projections under different hardware assumptions) as if they provide causal evidence about PoW\u2192PoS transitions, when they only model PoS energy under varying assumptions.\",\n    \"No acknowledgment of missing counterfactual: A true causal answer requires comparing the same network (or matched networks) under PoW vs PoS with identical validator sets and TPS. The answer never addresses whether such evidence exists or acknowledges its absence.\",\n    \"Overconfident quantitative claims: Statements like 'PoS energy per transaction is typically several orders of magnitude below Bitcoin's' are presented as causal effects when they come from comparing completely different networks with different validator counts, different TPS, and different security models.\",\n    \"The Bitcoin vs Ethereum/PoS comparisons cited involve massive differences in both validator count (Bitcoin miners vs Ethereum validators are structurally different) and throughput (Bitcoin ~7 TPS vs Ethereum ~15-30 TPS), yet these are used to support causal claims about the isolated effect of consensus mechanism.\"\n  ],\n  \"summary\": \"The answer fails because it confidently asserts large causal energy reductions from PoW\u2192PoS transitions while claiming to control for validator set size and TPS, but relies entirely on cross-chain comparisons and projections where these parameters are not actually matched. It never acknowledges that no study performs the required matched-parameter experiment, nor does it note that the confounds in existing comparisons prevent causal inference. The answer mistakes modeling exercises and cross-chain observational data for causal evidence.\",\n  \"verdict\": \"FAILED\"\n}\n```", "usage": {"input_tokens": 3597, "output_tokens": 705, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0}, "cost_usd": 0.021366, "latency_s": 17.980315375025384, "error": null, "run_id": "sample_001", "timestamp": "2026-06-02T20:55:09.259626+00:00"}
# {"kind": "verdict", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "verdict": "FAILED", "summary": "The answer fails because it confidently asserts large causal energy reductions from PoW\u2192PoS transitions while claiming to control for validator set size and TPS, but relies entirely on cross-chain comparisons and projections where these parameters are not actually matched. It never acknowledges that no study performs the required matched-parameter experiment, nor does it note that the confounds in existing comparisons prevent causal inference. The answer mistakes modeling exercises and cross-chain observational data for causal evidence.", "run_id": "sample_001", "timestamp": "2026-06-02T20:55:09.261189+00:00"}
# {"kind": "run_end", "totals": {"per_seed": [{"seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "status": "FAILED_FOUND", "cost": {"input_tokens": 3614, "output_tokens": 1550, "cache_creation_tokens": 0, "cache_read_tokens": 1777, "cost_usd": 0.0346251, "calls": 2}}], "grand_total": {"input_tokens": 3614, "output_tokens": 1550, "cache_creation_tokens": 0, "cache_read_tokens": 1777, "cost_usd": 0.0346251, "calls": 2}}, "run_id": "sample_001", "timestamp": "2026-06-02T20:55:09.261356+00:00"}

## Bloom’s taxonomy

- Remember: recall facts and basic concepts (define, deplication, list, memorize, repeat, state)
- Understand: explain ideas or concepts (classify, describe, discuss, explain, identify, locate, recognize, report, select, translate)
- **Apply: use information in new situations (execute, implement, solve, use, demonstrate, interpret, operate, schedule, sketch)**
- **Analyze: draw connections among ideas (differentiate, organize, relate, compare, contrast, distinguish, examine, experiment, question, test)**
- **Evaluate: justify a stand or decision (appraise, argue, defend, judge, select, support, value, critique, weigh)**
- Create: produce new or original work (design, assemble, construct, conjecture, develop, formulate, author, investigate)

In [17]:

# get starting and ending questions from all jsonl files in the directory (as pairs)
# {"kind": "run_start", "run_id": "sample_001", "timestamp": "2026-06-02T20:52:45.473449+00:00"}
# {"kind": "claude_call", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "purpose": "harder", "model": "claude-sonnet-4-5", "system": "You are an expert in constructing challenging research questions.\n\nGiven a seed question, produce an updated question designed to expose weaknesses in deep research systems. \n\nANSWERING SYSTEM PROFILE:\nThe answering system is an open \"deep research\" model that is trained to produce attributed long-form answers and whose ONLY tool is Semantic Scholar (\"S2\") search over academic papers. The system retrieves from a corpus of academic papers and synthesizes a cited report. Difficulty must not come from requiring sources outside this corpus. The system is good at surveying a single well-studied topic and producing a long well-structured report. It is bad at complex reasoning. \n\nOUTPUT FORMAT (valid JSON, no extra text):\n{\n  \"brainstorming\": \"<think about distinct strategies and reason about why they may or may not work>\",\n  \"chosen_strategy\": \"<name and justify the single most promising strategy>\",\n  \"updated_question\": \"<the rewritten question>\",\n  \"why_harder\": \"<explanation of why this question might be hard for a deep research system>\",\n  \"verification_criterion\": \"<one concrete, testable criterion for checking whether the answer is good>\"\n}\n\nRULES:\n- Avoid questions that can easily be answered by retrieving information.\n- The updated question should be hard to answer correctly, not just hard to retrieve \u2014 via higher-order thinking (analysis, comparison, evaluation, synthesis), a reasoning trap the system must catch (false premise, misconception, unanswerable claim), or an embedded constraint that changes what a correct answer must contain.\n- The updated question length should change by fewer than 15 words from the seed.\n- The verification criterion should be specific and checkable, not vague or aspirational. The criterion is checked by a judge who sees ONLY the question, the answer, and the answer's own cited sources \u2014 there is NO external answer key. So don't use hollow existence-counts like \"identify at least three implicit assumptions\" or \"name four categories of evidence.\" Anchor it to THIS question by naming the actual entities/claims at issue \u2014 never a generic template. \n- Select whichever strategy best fits THIS seed; do not default to the strategies shown in the examples below.\n\nEXAMPLE STRATEGIES TO CONSIDER:\n1. Require synthesis across 5+ sources or clearly disjoint domains (e.g., science + economics).\n2. Require synthesis across differing viewpoints, stakeholder incentives, or theoretical frameworks.\n3. Require multi-step reasoning, structured argumentation, or hierarchical planning.\n4. Require handling conflicting, incomplete, or low-quality evidence.\n5. Require correcting a hidden misconception or establishing key knowns before answering.\n6. Embed a specific context that changes the answer (e.g., \"explain to a policymaker with no ML background\").\n7. Make a question that is unanswerable by current research.\n\nHere are a few examples:\nSeed Question: What is pretraining-data deduplication?\n{\n\"brainstorming\": \"The current question can easily be answered by retrieving deduplication literature broadly. To make it harder, let's make the question unanswerable by asking a question that isn't answered by current literature.\",\n\"chosen_strategy\": \"Make the question unanswerable by asking a question that hasn't been resolved by current research.\",\n\"updated_question\": \"What is the causal contribution of pretraining-data deduplication to downstream reasoning, holding all else constant?\",\n\"why_harder\": \"A system can easily define deduplication, but it is much harder to determine its isolated causal effect on reasoning because existing studies do not cleanly vary only deduplication while holding other training factors fixed.\",\n\"verification_criterion\": \"Because no controlled study isolates this effect, the answer must EXPLICITLY state the question is unresolved by current research, name the specific missing evidence, and qualify any partial findings as correlational not causal. Fails if it asserts a confident causal answer or implies the literature resolves it.\"\n}\n\nSeed Question: What is the role of attention sparsity in efficient transformers?\n{\n\"brainstorming\": \"Surveying efficient-transformer literature would let the system define sparsity and list methods, so a pure synthesis question is too easy. One option is multi-step reasoning about FLOPs tradeoffs, but those numbers can be retrieved and quoted directly. A stronger option exploits that benchmark results for sparse attention genuinely conflict across papers: the system can locate both 'sparse wins' and 'sparse loses' results, but is bad at the reasoning needed to reconcile them via confounds.\",\n\"chosen_strategy\": \"Require reconciliation of conflicting evidence: force the system to explain WHY cited papers disagree rather than just report their results.\",\n\"updated_question\": \"When do sparse-attention transformers underperform dense baselines, and why do reported results conflict?\",\n\"why_harder\": \"A survey can enumerate sparse-attention methods and their headline numbers, but reconciling contradictory sparse-vs-dense comparisons requires identifying confounds (sequence length, task type, matched compute) that the papers themselves rarely make explicit, which is a reasoning task rather than a retrieval task.\",\n\"verification_criterion\": \"The answer must show that its OWN cited sources disagree (some reporting sparse >= dense, others sparse < dense) and attribute the conflict to at least one concrete confound such as sequence length, task type (long-range vs short-context), or matched compute budget. Fails if it issues a single uniform verdict, or if the papers it cites do not actually report conflicting sparse-vs-dense comparisons.\"\n}\n\nSeed Question: How does brown adipose tissue produce heat?\n{\n\"brainstorming\": \"The seed is a clean survey: retrieve BAT/UCP1 literature and summarize thermogenesis. A single-topic false premise (e.g. mislocating a function) fails, because if the corpus already frames it as a known misconception the system just retrieves the debunking. So embed a false CONJUNCTION whose refutation is not packaged anywhere: assert that (a) UCP1 drives ATP synthesis and (b) this powers shivering thermogenesis. Each underlying fact (UCP1 uncouples to make heat not ATP; BAT mediates NON-shivering thermogenesis) is documented separately as background, but no source refutes this composite because no one proposes it. A survey-strong, reasoning-weak system retrieves the facts yet writes fluently around the premise without noticing the contradiction.\",\n\"chosen_strategy\": \"False premise via conjunction of separately-documented facts.\",\n\"updated_question\": \"How does UCP1-driven ATP synthesis power shivering thermogenesis?\",\n\"why_harder\": \"Surveying BAT thermogenesis returns UCP1=uncoupling=heat and BAT=non-shivering as separate background facts, but nothing in the corpus is framed as refuting 'ATP-powered shivering.' A non-reasoning synthesis can therefore produce a fluent answer that silently honors the premise. Rejecting it requires conjoining two facts the literature never assembles against this claim: that UCP1 bypasses ATP synthase, and that it is the non-shivering pathway.\",\n\"verification_criterion\": \"The answer must reject BOTH embedded errors: (1) state that UCP1 uncouples oxidative phosphorylation and dissipates the proton gradient as heat rather than synthesizing ATP, i.e. it bypasses/short-circuits ATP synthase; and (2) state that UCP1/BAT mediates NON-shivering thermogenesis, which is distinct from and an alternative to shivering thermogenesis (skeletal-muscle contraction). Fails if it describes UCP1 as producing ATP, treats BAT/UCP1 as the mechanism of shivering, or answers fluently as though the premise were coherent.\"\n}\n", "messages": [{"role": "user", "content": "Seed question: \u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236"}], "response_text": "```json\n{\n  \"brainstorming\": \"The seed question '\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236' (blockchain consensus mechanisms) is extremely broad and invites a straightforward survey answer: define consensus, list PoW/PoS/PBFT/etc., describe their trade-offs. A pure synthesis across mechanisms is still too easy because the system can retrieve comparison tables directly. Multi-step reasoning about security proofs might work, but those are often formalized in papers. A stronger option: exploit that the Chinese-language academic corpus on blockchain is much sparser than English, yet S2 indexes primarily English papers. If I keep the question in Chinese, the system must either (a) find Chinese papers (few exist), (b) translate the question internally and search English (likely), or (c) fail. But difficulty from language mismatch violates the spirit since it's not about reasoning. So instead, rewrite in English but embed a reasoning trap: assert a false trade-off that SOUNDS plausible (e.g., 'finality speed always trades off against decentralization') which surveys repeat informally, but which counterexamples (like certain BFT-based designs) refute. However, those counterexamples are documented, so a retrieval-strong system might find them. A better trap: require reconciling WHY different consensus families (PoW vs. PoS vs. BFT) achieve 'security' via fundamentally different threat models (51% hashpower vs. 1/3 stake vs. Byzantine quorum), and ask which threat model applies to a HYBRID scenario the literature doesn't address. Actually, hybrid consensus (PoW+PoS) IS studied, so that's retrievable. Strongest option: pose an unanswerable causality question about a dimension (energy efficiency) where PoW vs. PoS comparisons exist but controlled experiments don't, forcing the system to recognize confounds or admit uncertainty rather than confidently claiming 'PoS is greener' as if it's proven causal.\",\n  \"chosen_strategy\": \"Unanswerable causality + conflicting evidence: Ask for the isolated causal effect of a consensus design choice where studies compare outcomes but don't control confounds, and where the question's framing assumes a clean answer exists.\",\n  \"updated_question\": \"What is the causal effect of switching from Proof-of-Work to Proof-of-Stake on network energy consumption, holding validator set size and transaction throughput constant?\",\n  \"why_harder\": \"Surveys can retrieve 'PoW uses more energy than PoS' as a headline claim, but no study cleanly isolates consensus mechanism while holding validator count and throughput fixed across both. Real-world PoW\u2192PoS transitions (Ethereum Merge) change multiple variables simultaneously (validator set, issuance, throughput targets), so confident causal attribution requires reasoning about confounds. A retrieval-strong system will cite energy comparisons but is unlikely to recognize they don't answer the controlled counterfactual, or to state the question is unanswerable with current evidence.\",\n  \"verification_criterion\": \"The answer must either (1) explicitly state that no study controls validator set size and throughput while varying only consensus mechanism, so the causal effect is not cleanly established, OR (2) if it claims a causal answer, cite a source that reports a matched-parameter experiment (same validator count, same TPS) comparing PoW vs. PoS energy. Fails if it confidently asserts 'PoS reduces energy by X%' using comparisons (e.g., Ethereum pre/post-Merge, Bitcoin vs. Cardano) where validator counts, throughput, or other parameters differ, without acknowledging those confounds prevent causal inference.\"\n}\n```", "usage": {"input_tokens": 17, "output_tokens": 845, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 1777}, "cost_usd": 0.013259100000000001, "latency_s": 26.107261416036636, "error": null, "run_id": "sample_001", "timestamp": "2026-06-02T20:53:11.675797+00:00"}
# {"kind": "research_call", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "url": "http://localhost:8007/ask", "request_json": {"question": "What is the causal effect of switching from Proof-of-Work to Proof-of-Stake on network energy consumption, holding validator set size and transaction throughput constant?"}, "response_status": 200, "response_body": {"answer_chars": 13667, "trace_present": true}, "latency_s": 99.59951554104919, "error": null, "run_id": "sample_001", "timestamp": "2026-06-02T20:54:51.276898+00:00"}
# {"kind": "claude_call", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "purpose": "judge", "model": "claude-sonnet-4-5", "system": null, "messages": [{"role": "user", "content": "You are an expert evaluator of deep research system outputs.\n\nYou will be given:\n- A research question\n- The verification criterion that defines what a good answer must do\n- The answer the research system produced\n\nYour job is to judge whether the answer satisfies the verification criterion, and to flag other issues you notice (factual errors, hallucinations, evasion, missing reasoning, structural problems, etc.) even if those issues are not part of the criterion.\n\nQUESTION:\nWhat is the causal effect of switching from Proof-of-Work to Proof-of-Stake on network energy consumption, holding validator set size and transaction throughput constant?\n\nVERIFICATION CRITERION:\nThe answer must either (1) explicitly state that no study controls validator set size and throughput while varying only consensus mechanism, so the causal effect is not cleanly established, OR (2) if it claims a causal answer, cite a source that reports a matched-parameter experiment (same validator count, same TPS) comparing PoW vs. PoS energy. Fails if it confidently asserts 'PoS reduces energy by X%' using comparisons (e.g., Ethereum pre/post-Merge, Bitcoin vs. Cardano) where validator counts, throughput, or other parameters differ, without acknowledging those confounds prevent causal inference.\n\nANSWER:\n<answer omitted: 13667 chars \u2014 see results file>\n\nOUTPUT FORMAT (valid JSON, no extra text):\n{\n  \"criterion_satisfied\": <true | false>,\n  \"criterion_reasoning\": \"<why the answer does or does not satisfy the criterion>\",\n  \"other_issues\": [\"<issue 1>\", \"<issue 2>\", ...],\n  \"summary\": \"<2-4 sentence overall summary of how the answer performed and what to push on next time>\",\n  \"verdict\": \"<PASSED | FAILED>\"\n}\n\nPASSED means: the criterion is satisfied AND there are no critical issues.\nFAILED means: the criterion is not satisfied OR there are serious problems (hallucinations, refusals, off-topic).\n"}], "response_text": "```json\n{\n  \"criterion_satisfied\": false,\n  \"criterion_reasoning\": \"The answer fails the verification criterion. It confidently asserts large energy reductions (10-100\u00d7, '10\u2013100\u00d7 lower than PoW', 'three orders of magnitude') from switching PoW to PoS while claiming to hold validator set size and TPS constant, but does NOT cite any study that actually performs a matched-parameter experiment with the same validator count AND same TPS comparing PoW vs PoS. Instead, it relies on: (1) cross-chain comparisons (Bitcoin vs PoS systems) where both validator counts and throughput differ dramatically, (2) Ethereum 2.0 beacon chain projections that model PoS only (not a PoW vs PoS comparison at matched parameters), and (3) methodological discussions about controlling for throughput/validators in principle, without citing an actual study that does so. The answer never acknowledges that these confounds prevent clean causal inference\u2014it instead presents these confounded comparisons as if they establish the causal effect, violating the criterion's requirement to either (1) state no such study exists, or (2) cite an actual matched-parameter experiment.\",\n  \"other_issues\": [\n    \"Misleading framing: The answer presents itself as providing 'a causal interpretation' and 'causal takeaway' while actually relying on observational cross-chain comparisons where multiple parameters differ simultaneously.\",\n    \"Conflation of evidence types: The answer treats modeling exercises (Ethereum 2.0 energy projections under different hardware assumptions) as if they provide causal evidence about PoW\u2192PoS transitions, when they only model PoS energy under varying assumptions.\",\n    \"No acknowledgment of missing counterfactual: A true causal answer requires comparing the same network (or matched networks) under PoW vs PoS with identical validator sets and TPS. The answer never addresses whether such evidence exists or acknowledges its absence.\",\n    \"Overconfident quantitative claims: Statements like 'PoS energy per transaction is typically several orders of magnitude below Bitcoin's' are presented as causal effects when they come from comparing completely different networks with different validator counts, different TPS, and different security models.\",\n    \"The Bitcoin vs Ethereum/PoS comparisons cited involve massive differences in both validator count (Bitcoin miners vs Ethereum validators are structurally different) and throughput (Bitcoin ~7 TPS vs Ethereum ~15-30 TPS), yet these are used to support causal claims about the isolated effect of consensus mechanism.\"\n  ],\n  \"summary\": \"The answer fails because it confidently asserts large causal energy reductions from PoW\u2192PoS transitions while claiming to control for validator set size and TPS, but relies entirely on cross-chain comparisons and projections where these parameters are not actually matched. It never acknowledges that no study performs the required matched-parameter experiment, nor does it note that the confounds in existing comparisons prevent causal inference. The answer mistakes modeling exercises and cross-chain observational data for causal evidence.\",\n  \"verdict\": \"FAILED\"\n}\n```", "usage": {"input_tokens": 3597, "output_tokens": 705, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0}, "cost_usd": 0.021366, "latency_s": 17.980315375025384, "error": null, "run_id": "sample_001", "timestamp": "2026-06-02T20:55:09.259626+00:00"}
# {"kind": "verdict", "seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "attempt": 1, "verdict": "FAILED", "summary": "The answer fails because it confidently asserts large causal energy reductions from PoW\u2192PoS transitions while claiming to control for validator set size and TPS, but relies entirely on cross-chain comparisons and projections where these parameters are not actually matched. It never acknowledges that no study performs the required matched-parameter experiment, nor does it note that the confounds in existing comparisons prevent causal inference. The answer mistakes modeling exercises and cross-chain observational data for causal evidence.", "run_id": "sample_001", "timestamp": "2026-06-02T20:55:09.261189+00:00"}
# {"kind": "run_end", "totals": {"per_seed": [{"seed": "\u533a\u5757\u94fe\u5171\u8bc6\u673a\u5236", "status": "FAILED_FOUND", "cost": {"input_tokens": 3614, "output_tokens": 1550, "cache_creation_tokens": 0, "cache_read_tokens": 1777, "cost_usd": 0.0346251, "calls": 2}}], "grand_total": {"input_tokens": 3614, "output_tokens": 1550, "cache_creation_tokens": 0, "cache_read_tokens": 1777, "cost_usd": 0.0346251, "calls": 2}}, "run_id": "sample_001", "timestamp": "2026-06-02T20:55:09.261356+00:00"}

# keep all generated questions; if it's not the last one for a seed, pair it with the next question for that seed, and label it PASSED, e.g., (seed, next_q, "PASSED"))
# if it is the last one for a seed, and the final verdict is FAILED_FOUND, label it FAILED, e.g., (seed, final_q, "FAILED")

question_pairs = []

for filename in os.listdir(DATA_DIR):
    if filename.endswith('.jsonl'):
        with open(os.path.join(DATA_DIR, filename), 'r') as f:
            for line in f:
                data = json.loads(line)
                if data['kind'] == 'claude_call' and data['purpose'] == 'harder':
                    seed = data['seed']
                    try:
                        response_json = json.loads(data['response_text'].strip('```json').strip('```'))
                    except json.JSONDecodeError:
                        print(data['response_text'])
                        continue
                    question_pairs.append([seed, response_json, "PASSED"])
                elif data['kind'] == 'verdict':
                    if data['verdict'] == 'FAILED':
                        question_pairs[-1][2] = "FAILED"

print(f"Total question pairs: {len(question_pairs)}")




```json
{
  "brainstorming": "The previous attempt required reconciling conflicting evidence about MILP vs SAT solver efficiency, and the system passed by identifying confounds like S-box complexity, constraint types, round numbers, etc. This means the system can successfully retrieve and organize contradictory results when they exist in the literature. To make it harder, I need a different strategy. Options: (1) Require synthesis across disjoint domains - but Square attack is a specialized cryptanalysis topic, hard to naturally bridge to unrelated fields. (2) Multi-step reasoning with hierarchical constraints - could work by requiring analysis of when improvements compose vs interfere. (3) False premise via conjunction - embed a claim that mixes real Square improvements with a mechanism that doesn't apply. (4) Unanswerable by current research - ask about a combination or analysis that hasn't been studied. (5) Require correcting a hidden misconception about how Square improvements wor

In [19]:
# save to file
with open('question_pairs.json', 'w') as f:
    json.dump(question_pairs, f, indent=2)

In [7]:
# Plot Bloom's-taxonomy distributions for: seed questions, PASSED updated questions,
# and FAILED updated questions. Run classify_blooms.py first to produce this file.
import json
from collections import Counter
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

BLOOM_ORDER = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]

with open("blooms_classified.json") as f:
    records = json.load(f)

# Three distributions:
#   Seed           -> bloom level of every seed question
#   Passed updated -> bloom level of updated questions whose run PASSED
#   Failed updated -> bloom level of updated questions whose run FAILED
seed_levels   = [r["seed_bloom_level"] for r in records]
passed_levels = [r["updated_bloom_level"] for r in records if r["result"] == "PASSED"]
failed_levels = [r["updated_bloom_level"] for r in records if r["result"] == "FAILED"]

groups = {
    f"Seed (n={len(seed_levels)})": seed_levels,
    f"Passed updated (n={len(passed_levels)})": passed_levels,
    f"Failed updated (n={len(failed_levels)})": failed_levels,
}

# Build a tidy frame of within-group percentages (drops any ERROR/unclassified).
rows = []
for label, levels in groups.items():
    counts = Counter(l for l in levels if l in BLOOM_ORDER)
    total = sum(counts.values()) or 1
    for b in BLOOM_ORDER:
        rows.append({"group": label, "level": b, "count": counts[b], "percent": 100 * counts[b] / total})
df = pd.DataFrame(rows)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5.5))
ax = sns.barplot(
    data=df, x="level", y="count", hue="group",
    order=BLOOM_ORDER, palette=["#7f7f7f", "#2ca02c", "#d62728"],
)
ax.set_xlabel("Bloom's taxonomy level (low → high cognitive demand)")
ax.set_ylabel("Number of questions in group")
ax.set_title("Bloom's taxonomy distribution: seed vs. passed vs. failed questions")
ax.legend(title="")
sns.despine()
plt.tight_layout()
plt.savefig("blooms_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


ImportError: Unable to import required dependencies:
numpy: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [ ]:
# density plots
plt.figure(figsize=(10, 5.5))
for label, levels in groups.items():
    # Convert levels to numeric for density plotting
    numeric_levels = [BLOOM_ORDER.index(l) for l in levels if l in BLOOM_ORDER]
    sns.kdeplot(numeric_levels, label=label, fill=True, alpha=0.3)
plt.xticks(range(len(BLOOM_ORDER)), BLOOM_ORDER)
plt.xlabel("Bloom's taxonomy level (low → high cognitive demand)")
plt.ylabel("Density")
plt.title("Bloom's taxonomy density: seed vs. passed vs. failed questions")
plt.legend(title="")
sns.despine()
plt.tight_layout()
plt.savefig("blooms_density.png", dpi=150, bbox_inches="tight")
plt.show()

NameError: name 'plt' is not defined

In [8]:
# Show passed and failed questions for "Evaluate" level
import json
from collections import Counter

BLOOM_ORDER = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]

with open("blooms_classified.json") as f:
    records = json.load(f)

# Three distributions:
#   Seed           -> bloom level of every seed question
#   Passed updated -> bloom level of updated questions whose run PASSED
#   Failed updated -> bloom level of updated questions whose run FAILED
passed_evaluate = [r for r in records if r["result"] == "PASSED" and r["updated_bloom_level"] == "Evaluate"]
failed_evaluate = [r for r in records if r["result"] == "FAILED" and r["updated_bloom_level"] == "Evaluate"]
print(f"Passed 'Evaluate' questions ({len(passed_evaluate)}):")
for r in passed_evaluate:
    print(f"- {r['updated_question']}")
print(f"\nFailed 'Evaluate' questions ({len(failed_evaluate)}):")
for r in failed_evaluate:
    print(f"- {r['updated_question']}")

Passed 'Evaluate' questions (11):
- Are the similarities between information foraging theory and exploratory search substantive or superficial at the theoretical level?
- When modeling ill-defined exploratory search tasks, do information foraging theory's optimality assumptions conflict with the sensemaking perspective, and if so, how should they be reconciled?
- When does prompt engineering provide the least marginal value across AI coding tools like Copilot, Cursor, Aider, and autonomous agents like Replit, and why?
- Do geopolitical-risk indices conflate distinct causal mechanisms, and does this measurement problem explain why trade effects are heterogeneous across studies?
- Which proposed smart-city solutions for sustainable tourism in Iran are infeasible given the country's documented infrastructure and technology-access constraints?
- Do existing wage inequality reviews systematically assess causal identification strategies, and if not, what should a methods-focused systematic r

In [9]:
# Show passed and failed questions for "Analyze" level
import json
from collections import Counter

BLOOM_ORDER = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]

with open("blooms_classified.json") as f:
    records = json.load(f)

# Three distributions:
#   Seed           -> bloom level of every seed question
#   Passed updated -> bloom level of updated questions whose run PASSED
#   Failed updated -> bloom level of updated questions whose run FAILED
passed_analyze = [r for r in records if r["result"] == "PASSED" and r["updated_bloom_level"] == "Analyze"]
failed_analyze = [r for r in records if r["result"] == "FAILED" and r["updated_bloom_level"] == "Analyze"]
print(f"Passed 'Analyze' questions ({len(passed_analyze)}):")
for r in passed_analyze:
    print(f"- {r['updated_question']}")
print(f"\nFailed 'Analyze' questions ({len(failed_analyze)}):")
for r in failed_analyze:
    print(f"- {r['updated_question']}")

Passed 'Analyze' questions (21):
- Why do static thermal conductivity values misrepresent the building physics performance of hygroscopic materials like hempcrete and straw bale?
- When do xenotransplantation techniques fail due to hyperacute rejection, and why do reported survival outcomes conflict?
- 对于Square攻击的自动化搜索，为什么不同研究对MILP与SAT求解器的效率评价存在矛盾？请用中文回答。
- What distinguishes rehabilitation protocols for chronic ankle instability versus acute lateral ankle sprains in pediatric patients?
- Why do Latin American and Italian scholars diverge in theorizing estado ampliado's relationship to civil society autonomy?
- Comment les critères de convergence macroéconomique de l'UEMOA modifient-ils les déterminants actuariels classiques de la viabilité du FNRB par rapport à un fonds de retraite hors contrainte budgétaire supranationale ?
- Kiến thức về các biện pháp phòng tránh thai ở người dân tộc thiểu số làm giảm tỷ lệ mang thai ngoài ý muốn như thế nào?
- Why do flexible marine risers that ena